In [9]:
import numpy as np
from numpy.random import default_rng, SeedSequence
from numpy.random import Generator as _NpGen, RandomState as _RS
import functions_list_260305 as functions_list
import summary_stats_elms_260305 as ss
import hashlib
import time
import matplotlib.pyplot as plt


start = time.perf_counter()

# ============================================================================
# SETUP: RNG and parameters
# ============================================================================
rng = np.random.default_rng(123)
core_params_num = 2  # core params: R0 and sigma

# fixed parameters
DurationSimulation = 20.0     # years: 20.0
Nstrains = 40       # number of strains: 42
omega = 0.2     # immunity cross strains: 0.1
x = 10.0        #
Cperweek = 34.53    #
Nagents = 1000      # number of agents
alpha = 0.007        # migration rate: 3.0
AgeDeath = 71.0     # life expectancy

# ============================================================================
# BUILD FIXED PARAMS ARRAY
# ============================================================================
if core_params_num == 2:
    Dimmunity = 10.0 * 52.14  # weeks: 0.5 * 52.14
    fixed_params = np.array([DurationSimulation, Nstrains, Dimmunity, omega,
                         x, Cperweek, Nagents, alpha,
                         AgeDeath], dtype=float)
elif core_params_num == 3:
    fixed_params = np.array([DurationSimulation, Nstrains, omega, x,
                             Cperweek, Nagents, alpha, AgeDeath], dtype=float)
else:
    raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: build_params
# ============================================================================
def build_params(theta, fixed_params, core_params_num):
    theta = np.asarray(theta, float).ravel()
    if theta.size != core_params_num:
        raise ValueError(f"theta must be length-{core_params_num}, got {np.shape(theta)}")
    if core_params_num == 2:
        R0, sigma = float(theta[0]), float(theta[1])
        return np.array([fixed_params[0], fixed_params[1], fixed_params[2], sigma,
                         fixed_params[3], fixed_params[4], fixed_params[5], fixed_params[6],
                         fixed_params[7], fixed_params[8], R0
                         ], dtype=float)
    elif core_params_num == 3:
        R0, sigma, Dimmunity = float(theta[0]), float(theta[1]), float(theta[2])
        return np.array([fixed_params[0], fixed_params[1], Dimmunity, sigma,
                         fixed_params[2], fixed_params[3], fixed_params[4], fixed_params[5],
                         fixed_params[6], fixed_params[7], R0
                         ], dtype=float)
    else:
        raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: seed_from_theta
# ============================================================================
def seed_from_theta(theta, master_seed: int = 123):
    th = np.asarray(theta, np.float64).ravel()
    b  = th.tobytes() + np.uint64(master_seed).tobytes()
    return int.from_bytes(hashlib.sha1(b).digest()[:8], 'little')

# ============================================================================
# FUNCTION: simulate_prevalence_v5_numba
# ============================================================================
def simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed):
    seed = seed_from_theta(theta, master_seed=seed)
    rng = default_rng(seed)
    params = build_params(theta, fixed_params, core_params_num)
    AC, IMM, _ = functions_list.initialise_agents_v5(params, rng=rng)
    SSPrev_selected, SSPrev, AIBKS = functions_list.simulator_v5_numba(
        AC, IMM, params, 0, 1, seed=seed
    )
    return SSPrev_selected.astype(float)

# ============================================================================
# FUNCTION: summary_stats
# ============================================================================
def summary_stats(series_2d):
    y = np.asarray(series_2d, float).ravel()
    avg_prev_obs = ss.avg_prev_numpy(series_2d)
    var_prev_obs = np.sqrt(ss.var_prev_numpy(series_2d))
    avg_npmi_obs = ss.avg_npmi_numpy(series_2d)
    div_all_isolates_obs = ss.div_all_isolates_numpy(series_2d)
    return np.array(
        [avg_prev_obs, var_prev_obs, avg_npmi_obs, div_all_isolates_obs], float)

# ============================================================================
# GENERATE SYNTHETIC DATA
# ============================================================================
if core_params_num == 2:
    _Tdry = simulate_prevalence_v5_numba(np.array([2.0, 0.4], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([2.0, 0.4], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
elif core_params_num == 3:
    _Tdry = simulate_prevalence_v5_numba(np.array([2.0, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([2.0, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
else:
    raise ValueError('Invalid core params num')

y_obs_array = _Tdry
print("y_obs_array.mean:", y_obs_array.mean())
# ============================================================================
# LOAD POSTERIOR SAMPLES
# ============================================================================
csv_path_R0 = "../../experimental_data/from_260312/R0_samps_2params_R02p0.csv"
csv_path_sigma = "../../experimental_data/from_260312/sigma_samps_2params_R02p0.csv"
total_length = 2000
R0_samps = np.loadtxt(csv_path_R0, delimiter=",")
sigma_samps = np.loadtxt(csv_path_sigma, delimiter=",")
R0_samps = np.asarray(R0_samps, dtype=float).ravel()
sigma_samps = np.asarray(sigma_samps, dtype=float).ravel()
theta_samps = np.column_stack((R0_samps, sigma_samps))
theta_samps = theta_samps[np.isfinite(theta_samps).all(axis=1)]
theta_samps = theta_samps[:total_length]
R0_samps = theta_samps[:, 0]
sigma_samps = theta_samps[:, 1]
print("R0", R0_samps.shape, R0_samps[:10])
print("sigma", sigma_samps.shape, sigma_samps[:10])
print("theta", theta_samps.shape, theta_samps[:10, :])

# ============================================================================
# FUNCTION: simulate_at_obs
# ============================================================================
def simulate_at_obs(theta, seed):
    rng = np.random.default_rng(seed)
    Y = simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed=seed)
    return Y

# ============================================================================
# GENERATE POSTERIOR PREDICTIVE CHECKS
# ============================================================================
ppc = []
for m, th in enumerate(theta_samps):
    Y_m = simulate_at_obs(th, seed=123)
    ppc.append(Y_m)
ppc = np.stack(ppc, axis=0)

# ============================================================================
# COMPUTE PREDICTIVE SUMMARIES
# ============================================================================
pred_mean = ppc.mean(axis=0) 
pred_lo, pred_hi = np.quantile(ppc, [0.05, 0.95], axis=0)

# ============================================================================
# FUNCTION: rmse
# ============================================================================
def rmse(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    d = A[mask] - B[mask]
    return float(np.sqrt(np.mean(d*d)))

# ============================================================================
# FUNCTION: mae
# ============================================================================
def mae(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    return float(np.mean(np.abs(A[mask] - B[mask])))

# ============================================================================
# COMPUTE ERROR METRICS
# ============================================================================
rmse_mean = rmse(pred_mean, y_obs_array)
mae_mean  = mae(pred_mean, y_obs_array)
print("rmse:", rmse_mean)
print("mae:", mae_mean)

# ============================================================================
# COMPUTE COVERAGE
# ============================================================================
inside = (y_obs_array >= pred_lo) & (y_obs_array <= pred_hi)
coverage = float(np.mean(inside[np.isfinite(y_obs_array)]))
print("coverage:", coverage)

end = time.perf_counter()
print(f"Elapsed: {end - start:.4f} s")

# ============================================================================
# FUNCTION: heatmap
# ============================================================================
def heatmap(M, title, save_path=None):
    plt.figure(figsize=(7, 4.5))
    plt.imshow(M, aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.xlabel("time index")
    plt.ylabel("strain")
    plt.tight_layout()
    plt.savefig(f"{save_path}{title}.png", dpi=300, bbox_inches="tight")
    plt.show()

# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
heatmap(y_obs_array, "Observed", save_path="../../figures/from_260312/ppc/sigma0p4/R02p0/")
heatmap(pred_mean, "PPC mean", save_path="../../figures/from_260312/ppc/sigma0p4/R02p0/")
heatmap(pred_mean - y_obs_array, "Error (mean - obs)", save_path="../../figures/from_260312/ppc/sigma0p4/R02p0/")

# ============================================================================
# ADDITIONAL DIAGNOSTICS (OPTIONAL)
# ============================================================================
# Uncertainty heatmap
heatmap(pred_hi - pred_lo, "90% CI Width (Uncertainty)", save_path="../../figures/from_260312/ppc/sigma0p4/R02p0/")

# Coverage map
heatmap(inside.astype(float), "Coverage Map (1=inside, 0=outside)", save_path="../../figures/from_260312/ppc/sigma0p4/R02p0/")

# Strain-specific metrics
strain_coverage = inside.mean(axis=1)
strain_mae = np.abs(pred_mean - y_obs_array).mean(axis=1)
print("Worst strain coverage:", strain_coverage.min())
print("Best strain coverage:", strain_coverage.max())
print("Worst strain MAE:", strain_mae.max())

# Time-specific metrics
time_coverage = inside.mean(axis=0)
time_mae = np.abs(pred_mean - y_obs_array).mean(axis=0)
print("Worst time coverage:", time_coverage.min())
print("Best time coverage:", time_coverage.max())
print("Worst time MAE:", time_mae.max())

# ============================================================================
# SUMMARY REPORT
# ============================================================================
print("=" * 60)
print("POSTERIOR PREDICTIVE CHECK RESULTS")
print("=" * 60)
print(f"Ground truth:      R0 = 2.0, sigma = 0.4")
print(f"Posterior samples: {len(theta_samps)}")
print(f"R0 posterior:      {R0_samps.mean():.3f} ± {R0_samps.std():.3f}")
print(f"sigma posterior:   {sigma_samps.mean():.3f} ± {sigma_samps.std():.3f}")
print("-" * 60)
print(f"RMSE:              {rmse_mean:.6f}")
print(f"MAE:               {mae_mean:.6f}")
print(f"90% CI Coverage:   {coverage*100:.2f}%")
print(f"Expected Coverage: 90.00%")
print("-" * 60)
if 0.88 <= coverage <= 0.92:
    print("Status: ✓ Well-calibrated uncertainty")
elif coverage < 0.88:
    print("Status: ⚠ Under-coverage (model overconfident)")
else:
    print("Status: ⚠ Over-coverage (model underconfident)")
print("=" * 60)

T's size 920
True True
y_obs_array.mean: 0.9891304347826086
R0 (2000,) [1.88143949 3.96586821 1.9262225  1.73735195 2.98680737 4.34040939
 2.94256561 2.71932128 1.71650703 3.23699871]
sigma (2000,) [0.34749745 0.70395216 0.33272319 0.21191593 0.57465003 0.693193
 0.61529212 0.44224171 0.22423573 0.543045  ]
theta (2000, 2) [[1.88143949 0.34749745]
 [3.96586821 0.70395216]
 [1.9262225  0.33272319]
 [1.73735195 0.21191593]
 [2.98680737 0.57465003]
 [4.34040939 0.693193  ]
 [2.94256561 0.61529212]
 [2.71932128 0.44224171]
 [1.71650703 0.22423573]
 [3.23699871 0.543045  ]]


SystemError: CPUDispatcher(<function _simulator_v4_core at 0x124157c10>) returned a result with an error set

In [10]:
import numpy as np
from numpy.random import default_rng, SeedSequence
from numpy.random import Generator as _NpGen, RandomState as _RS
import functions_list_260305 as functions_list
import summary_stats_elms_260305 as ss
import hashlib
import time
import matplotlib.pyplot as plt


start = time.perf_counter()

# ============================================================================
# SETUP: RNG and parameters
# ============================================================================
rng = np.random.default_rng(123)
core_params_num = 2  # core params: R0 and sigma

# fixed parameters
DurationSimulation = 20.0     # years: 20.0
Nstrains = 40       # number of strains: 42
omega = 0.2     # immunity cross strains: 0.1
x = 10.0        #
Cperweek = 34.53    #
Nagents = 1000      # number of agents
alpha = 0.007        # migration rate: 3.0
AgeDeath = 71.0     # life expectancy

# ============================================================================
# BUILD FIXED PARAMS ARRAY
# ============================================================================
if core_params_num == 2:
    Dimmunity = 10.0 * 52.14  # weeks: 0.5 * 52.14
    fixed_params = np.array([DurationSimulation, Nstrains, Dimmunity, omega,
                         x, Cperweek, Nagents, alpha,
                         AgeDeath], dtype=float)
elif core_params_num == 3:
    fixed_params = np.array([DurationSimulation, Nstrains, omega, x,
                             Cperweek, Nagents, alpha, AgeDeath], dtype=float)
else:
    raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: build_params
# ============================================================================
def build_params(theta, fixed_params, core_params_num):
    theta = np.asarray(theta, float).ravel()
    if theta.size != core_params_num:
        raise ValueError(f"theta must be length-{core_params_num}, got {np.shape(theta)}")
    if core_params_num == 2:
        R0, sigma = float(theta[0]), float(theta[1])
        return np.array([fixed_params[0], fixed_params[1], fixed_params[2], sigma,
                         fixed_params[3], fixed_params[4], fixed_params[5], fixed_params[6],
                         fixed_params[7], fixed_params[8], R0
                         ], dtype=float)
    elif core_params_num == 3:
        R0, sigma, Dimmunity = float(theta[0]), float(theta[1]), float(theta[2])
        return np.array([fixed_params[0], fixed_params[1], Dimmunity, sigma,
                         fixed_params[2], fixed_params[3], fixed_params[4], fixed_params[5],
                         fixed_params[6], fixed_params[7], R0
                         ], dtype=float)
    else:
        raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: seed_from_theta
# ============================================================================
def seed_from_theta(theta, master_seed: int = 123):
    th = np.asarray(theta, np.float64).ravel()
    b  = th.tobytes() + np.uint64(master_seed).tobytes()
    return int.from_bytes(hashlib.sha1(b).digest()[:8], 'little')

# ============================================================================
# FUNCTION: simulate_prevalence_v5_numba
# ============================================================================
def simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed):
    seed = seed_from_theta(theta, master_seed=seed)
    rng = default_rng(seed)
    params = build_params(theta, fixed_params, core_params_num)
    AC, IMM, _ = functions_list.initialise_agents_v5(params, rng=rng)
    SSPrev_selected, SSPrev, AIBKS = functions_list.simulator_v5_numba(
        AC, IMM, params, 0, 1, seed=seed
    )
    return SSPrev_selected.astype(float)

# ============================================================================
# FUNCTION: summary_stats
# ============================================================================
def summary_stats(series_2d):
    y = np.asarray(series_2d, float).ravel()
    avg_prev_obs = ss.avg_prev_numpy(series_2d)
    var_prev_obs = np.sqrt(ss.var_prev_numpy(series_2d))
    avg_npmi_obs = ss.avg_npmi_numpy(series_2d)
    div_all_isolates_obs = ss.div_all_isolates_numpy(series_2d)
    return np.array(
        [avg_prev_obs, var_prev_obs, avg_npmi_obs, div_all_isolates_obs], float)

# ============================================================================
# GENERATE SYNTHETIC DATA
# ============================================================================
if core_params_num == 2:
    _Tdry = simulate_prevalence_v5_numba(np.array([2.5, 0.4], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([2.5, 0.4], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
elif core_params_num == 3:
    _Tdry = simulate_prevalence_v5_numba(np.array([2.5, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([2.5, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
else:
    raise ValueError('Invalid core params num')

y_obs_array = _Tdry
print("y_obs_array.mean:", y_obs_array.mean())
# ============================================================================
# LOAD POSTERIOR SAMPLES
# ============================================================================
csv_path_R0 = "../../experimental_data/from_260312/R0_samps_2params_R02p5.csv"
csv_path_sigma = "../../experimental_data/from_260312/sigma_samps_2params_R02p5.csv"
total_length = 2000
R0_samps = np.loadtxt(csv_path_R0, delimiter=",")
sigma_samps = np.loadtxt(csv_path_sigma, delimiter=",")
R0_samps = np.asarray(R0_samps, dtype=float).ravel()
sigma_samps = np.asarray(sigma_samps, dtype=float).ravel()
theta_samps = np.column_stack((R0_samps, sigma_samps))
theta_samps = theta_samps[np.isfinite(theta_samps).all(axis=1)]
theta_samps = theta_samps[:total_length]
R0_samps = theta_samps[:, 0]
sigma_samps = theta_samps[:, 1]
print("R0", R0_samps.shape, R0_samps[:10])
print("sigma", sigma_samps.shape, sigma_samps[:10])
print("theta", theta_samps.shape, theta_samps[:10, :])

# ============================================================================
# FUNCTION: simulate_at_obs
# ============================================================================
def simulate_at_obs(theta, seed):
    rng = np.random.default_rng(seed)
    Y = simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed=seed)
    return Y

# ============================================================================
# GENERATE POSTERIOR PREDICTIVE CHECKS
# ============================================================================
ppc = []
for m, th in enumerate(theta_samps):
    Y_m = simulate_at_obs(th, seed=123)
    ppc.append(Y_m)
ppc = np.stack(ppc, axis=0)

# ============================================================================
# COMPUTE PREDICTIVE SUMMARIES
# ============================================================================
pred_mean = ppc.mean(axis=0) 
pred_lo, pred_hi = np.quantile(ppc, [0.05, 0.95], axis=0)

# ============================================================================
# FUNCTION: rmse
# ============================================================================
def rmse(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    d = A[mask] - B[mask]
    return float(np.sqrt(np.mean(d*d)))

# ============================================================================
# FUNCTION: mae
# ============================================================================
def mae(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    return float(np.mean(np.abs(A[mask] - B[mask])))

# ============================================================================
# COMPUTE ERROR METRICS
# ============================================================================
rmse_mean = rmse(pred_mean, y_obs_array)
mae_mean  = mae(pred_mean, y_obs_array)
print("rmse:", rmse_mean)
print("mae:", mae_mean)

# ============================================================================
# COMPUTE COVERAGE
# ============================================================================
inside = (y_obs_array >= pred_lo) & (y_obs_array <= pred_hi)
coverage = float(np.mean(inside[np.isfinite(y_obs_array)]))
print("coverage:", coverage)

end = time.perf_counter()
print(f"Elapsed: {end - start:.4f} s")

# ============================================================================
# FUNCTION: heatmap
# ============================================================================
def heatmap(M, title, save_path=None):
    plt.figure(figsize=(7, 4.5))
    plt.imshow(M, aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.xlabel("time index")
    plt.ylabel("strain")
    plt.tight_layout()
    plt.savefig(f"{save_path}{title}.png", dpi=300, bbox_inches="tight")
    plt.show()

# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
heatmap(y_obs_array, "Observed", save_path="../../figures/from_260312/ppc/sigma0p4/R02p5/")
heatmap(pred_mean, "PPC mean", save_path="../../figures/from_260312/ppc/sigma0p4/R02p5/")
heatmap(pred_mean - y_obs_array, "Error (mean - obs)", save_path="../../figures/from_260312/ppc/sigma0p4/R02p5/")

# ============================================================================
# ADDITIONAL DIAGNOSTICS (OPTIONAL)
# ============================================================================
# Uncertainty heatmap
heatmap(pred_hi - pred_lo, "90% CI Width (Uncertainty)", save_path="../../figures/from_260312/ppc/sigma0p4/R02p5/")

# Coverage map
heatmap(inside.astype(float), "Coverage Map (1=inside, 0=outside)", save_path="../../figures/from_260312/ppc/sigma0p4/R02p5/")

# Strain-specific metrics
strain_coverage = inside.mean(axis=1)
strain_mae = np.abs(pred_mean - y_obs_array).mean(axis=1)
print("Worst strain coverage:", strain_coverage.min())
print("Best strain coverage:", strain_coverage.max())
print("Worst strain MAE:", strain_mae.max())

# Time-specific metrics
time_coverage = inside.mean(axis=0)
time_mae = np.abs(pred_mean - y_obs_array).mean(axis=0)
print("Worst time coverage:", time_coverage.min())
print("Best time coverage:", time_coverage.max())
print("Worst time MAE:", time_mae.max())

# ============================================================================
# SUMMARY REPORT
# ============================================================================
print("=" * 60)
print("POSTERIOR PREDICTIVE CHECK RESULTS")
print("=" * 60)
print(f"Ground truth:      R0 = 2.5, sigma = 0.4")
print(f"Posterior samples: {len(theta_samps)}")
print(f"R0 posterior:      {R0_samps.mean():.3f} ± {R0_samps.std():.3f}")
print(f"sigma posterior:   {sigma_samps.mean():.3f} ± {sigma_samps.std():.3f}")
print("-" * 60)
print(f"RMSE:              {rmse_mean:.6f}")
print(f"MAE:               {mae_mean:.6f}")
print(f"90% CI Coverage:   {coverage*100:.2f}%")
print(f"Expected Coverage: 90.00%")
print("-" * 60)
if 0.88 <= coverage <= 0.92:
    print("Status: ✓ Well-calibrated uncertainty")
elif coverage < 0.88:
    print("Status: ⚠ Under-coverage (model overconfident)")
else:
    print("Status: ⚠ Over-coverage (model underconfident)")
print("=" * 60)

T's size 920
True True
y_obs_array.mean: 2.0673913043478263
R0 (2000,) [3.96586821 4.67440197 4.34040939 3.26435635 2.71932128 3.23699871
 2.48260466 2.08817901 2.16859667 2.83114371]
sigma (2000,) [0.70395216 0.7004272  0.693193   0.54995229 0.44224171 0.543045
 0.46284407 0.34409676 0.30254035 0.51506412]
theta (2000, 2) [[3.96586821 0.70395216]
 [4.67440197 0.7004272 ]
 [4.34040939 0.693193  ]
 [3.26435635 0.54995229]
 [2.71932128 0.44224171]
 [3.23699871 0.543045  ]
 [2.48260466 0.46284407]
 [2.08817901 0.34409676]
 [2.16859667 0.30254035]
 [2.83114371 0.51506412]]


SystemError: CPUDispatcher(<function _simulator_v4_core at 0x124157c10>) returned a result with an error set

In [11]:
import numpy as np
from numpy.random import default_rng, SeedSequence
from numpy.random import Generator as _NpGen, RandomState as _RS
import functions_list_260305 as functions_list
import summary_stats_elms_260305 as ss
import hashlib
import time
import matplotlib.pyplot as plt


start = time.perf_counter()

# ============================================================================
# SETUP: RNG and parameters
# ============================================================================
rng = np.random.default_rng(123)
core_params_num = 2  # core params: R0 and sigma

# fixed parameters
DurationSimulation = 20.0     # years: 20.0
Nstrains = 40       # number of strains: 42
omega = 0.2     # immunity cross strains: 0.1
x = 10.0        #
Cperweek = 34.53    #
Nagents = 1000      # number of agents
alpha = 0.007        # migration rate: 3.0
AgeDeath = 71.0     # life expectancy

# ============================================================================
# BUILD FIXED PARAMS ARRAY
# ============================================================================
if core_params_num == 2:
    Dimmunity = 10.0 * 52.14  # weeks: 0.5 * 52.14
    fixed_params = np.array([DurationSimulation, Nstrains, Dimmunity, omega,
                         x, Cperweek, Nagents, alpha,
                         AgeDeath], dtype=float)
elif core_params_num == 3:
    fixed_params = np.array([DurationSimulation, Nstrains, omega, x,
                             Cperweek, Nagents, alpha, AgeDeath], dtype=float)
else:
    raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: build_params
# ============================================================================
def build_params(theta, fixed_params, core_params_num):
    theta = np.asarray(theta, float).ravel()
    if theta.size != core_params_num:
        raise ValueError(f"theta must be length-{core_params_num}, got {np.shape(theta)}")
    if core_params_num == 2:
        R0, sigma = float(theta[0]), float(theta[1])
        return np.array([fixed_params[0], fixed_params[1], fixed_params[2], sigma,
                         fixed_params[3], fixed_params[4], fixed_params[5], fixed_params[6],
                         fixed_params[7], fixed_params[8], R0
                         ], dtype=float)
    elif core_params_num == 3:
        R0, sigma, Dimmunity = float(theta[0]), float(theta[1]), float(theta[2])
        return np.array([fixed_params[0], fixed_params[1], Dimmunity, sigma,
                         fixed_params[2], fixed_params[3], fixed_params[4], fixed_params[5],
                         fixed_params[6], fixed_params[7], R0
                         ], dtype=float)
    else:
        raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: seed_from_theta
# ============================================================================
def seed_from_theta(theta, master_seed: int = 123):
    th = np.asarray(theta, np.float64).ravel()
    b  = th.tobytes() + np.uint64(master_seed).tobytes()
    return int.from_bytes(hashlib.sha1(b).digest()[:8], 'little')

# ============================================================================
# FUNCTION: simulate_prevalence_v5_numba
# ============================================================================
def simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed):
    seed = seed_from_theta(theta, master_seed=seed)
    rng = default_rng(seed)
    params = build_params(theta, fixed_params, core_params_num)
    AC, IMM, _ = functions_list.initialise_agents_v5(params, rng=rng)
    SSPrev_selected, SSPrev, AIBKS = functions_list.simulator_v5_numba(
        AC, IMM, params, 0, 1, seed=seed
    )
    return SSPrev_selected.astype(float)

# ============================================================================
# FUNCTION: summary_stats
# ============================================================================
def summary_stats(series_2d):
    y = np.asarray(series_2d, float).ravel()
    avg_prev_obs = ss.avg_prev_numpy(series_2d)
    var_prev_obs = np.sqrt(ss.var_prev_numpy(series_2d))
    avg_npmi_obs = ss.avg_npmi_numpy(series_2d)
    div_all_isolates_obs = ss.div_all_isolates_numpy(series_2d)
    return np.array(
        [avg_prev_obs, var_prev_obs, avg_npmi_obs, div_all_isolates_obs], float)

# ============================================================================
# GENERATE SYNTHETIC DATA
# ============================================================================
if core_params_num == 2:
    _Tdry = simulate_prevalence_v5_numba(np.array([3.0, 0.4], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([3.0, 0.4], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
elif core_params_num == 3:
    _Tdry = simulate_prevalence_v5_numba(np.array([3.0, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([3.0, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
else:
    raise ValueError('Invalid core params num')

y_obs_array = _Tdry
print("y_obs_array.mean:", y_obs_array.mean())
# ============================================================================
# LOAD POSTERIOR SAMPLES
# ============================================================================
csv_path_R0 = "../../experimental_data/from_260312/R0_samps_2params_R03p0.csv"
csv_path_sigma = "../../experimental_data/from_260312/sigma_samps_2params_R03p0.csv"
total_length = 2000
R0_samps = np.loadtxt(csv_path_R0, delimiter=",")
sigma_samps = np.loadtxt(csv_path_sigma, delimiter=",")
R0_samps = np.asarray(R0_samps, dtype=float).ravel()
sigma_samps = np.asarray(sigma_samps, dtype=float).ravel()
theta_samps = np.column_stack((R0_samps, sigma_samps))
theta_samps = theta_samps[np.isfinite(theta_samps).all(axis=1)]
theta_samps = theta_samps[:total_length]
R0_samps = theta_samps[:, 0]
sigma_samps = theta_samps[:, 1]
print("R0", R0_samps.shape, R0_samps[:10])
print("sigma", sigma_samps.shape, sigma_samps[:10])
print("theta", theta_samps.shape, theta_samps[:10, :])

# ============================================================================
# FUNCTION: simulate_at_obs
# ============================================================================
def simulate_at_obs(theta, seed):
    rng = np.random.default_rng(seed)
    Y = simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed=seed)
    return Y

# ============================================================================
# GENERATE POSTERIOR PREDICTIVE CHECKS
# ============================================================================
ppc = []
for m, th in enumerate(theta_samps):
    Y_m = simulate_at_obs(th, seed=123)
    ppc.append(Y_m)
ppc = np.stack(ppc, axis=0)

# ============================================================================
# COMPUTE PREDICTIVE SUMMARIES
# ============================================================================
pred_mean = ppc.mean(axis=0) 
pred_lo, pred_hi = np.quantile(ppc, [0.05, 0.95], axis=0)

# ============================================================================
# FUNCTION: rmse
# ============================================================================
def rmse(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    d = A[mask] - B[mask]
    return float(np.sqrt(np.mean(d*d)))

# ============================================================================
# FUNCTION: mae
# ============================================================================
def mae(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    return float(np.mean(np.abs(A[mask] - B[mask])))

# ============================================================================
# COMPUTE ERROR METRICS
# ============================================================================
rmse_mean = rmse(pred_mean, y_obs_array)
mae_mean  = mae(pred_mean, y_obs_array)
print("rmse:", rmse_mean)
print("mae:", mae_mean)

# ============================================================================
# COMPUTE COVERAGE
# ============================================================================
inside = (y_obs_array >= pred_lo) & (y_obs_array <= pred_hi)
coverage = float(np.mean(inside[np.isfinite(y_obs_array)]))
print("coverage:", coverage)

end = time.perf_counter()
print(f"Elapsed: {end - start:.4f} s")

# ============================================================================
# FUNCTION: heatmap
# ============================================================================
def heatmap(M, title, save_path=None):
    plt.figure(figsize=(7, 4.5))
    plt.imshow(M, aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.xlabel("time index")
    plt.ylabel("strain")
    plt.tight_layout()
    plt.savefig(f"{save_path}{title}.png", dpi=300, bbox_inches="tight")
    plt.show()

# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
heatmap(y_obs_array, "Observed", save_path="../../figures/from_260312/ppc/sigma0p4/R03p0/")
heatmap(pred_mean, "PPC mean", save_path="../../figures/from_260312/ppc/sigma0p4/R03p0/")
heatmap(pred_mean - y_obs_array, "Error (mean - obs)", save_path="../../figures/from_260312/ppc/sigma0p4/R03p0/")

# ============================================================================
# ADDITIONAL DIAGNOSTICS (OPTIONAL)
# ============================================================================
# Uncertainty heatmap
heatmap(pred_hi - pred_lo, "90% CI Width (Uncertainty)", save_path="../../figures/from_260312/ppc/sigma0p4/R03p0/")

# Coverage map
heatmap(inside.astype(float), "Coverage Map (1=inside, 0=outside)", save_path="../../figures/from_260312/ppc/sigma0p4/R03p0/")

# Strain-specific metrics
strain_coverage = inside.mean(axis=1)
strain_mae = np.abs(pred_mean - y_obs_array).mean(axis=1)
print("Worst strain coverage:", strain_coverage.min())
print("Best strain coverage:", strain_coverage.max())
print("Worst strain MAE:", strain_mae.max())

# Time-specific metrics
time_coverage = inside.mean(axis=0)
time_mae = np.abs(pred_mean - y_obs_array).mean(axis=0)
print("Worst time coverage:", time_coverage.min())
print("Best time coverage:", time_coverage.max())
print("Worst time MAE:", time_mae.max())

# ============================================================================
# SUMMARY REPORT
# ============================================================================
print("=" * 60)
print("POSTERIOR PREDICTIVE CHECK RESULTS")
print("=" * 60)
print(f"Ground truth:      R0 = 3.0, sigma = 0.4")
print(f"Posterior samples: {len(theta_samps)}")
print(f"R0 posterior:      {R0_samps.mean():.3f} ± {R0_samps.std():.3f}")
print(f"sigma posterior:   {sigma_samps.mean():.3f} ± {sigma_samps.std():.3f}")
print("-" * 60)
print(f"RMSE:              {rmse_mean:.6f}")
print(f"MAE:               {mae_mean:.6f}")
print(f"90% CI Coverage:   {coverage*100:.2f}%")
print(f"Expected Coverage: 90.00%")
print("-" * 60)
if 0.88 <= coverage <= 0.92:
    print("Status: ✓ Well-calibrated uncertainty")
elif coverage < 0.88:
    print("Status: ⚠ Under-coverage (model overconfident)")
else:
    print("Status: ⚠ Over-coverage (model underconfident)")
print("=" * 60)

T's size 920
True True
y_obs_array.mean: 2.7271739130434782
R0 (2000,) [3.56485228 4.70733526 1.92168994 3.48350921 3.87936542 3.83044543
 3.1491516  1.89563379 3.79624839 2.85325583]
sigma (2000,) [0.39597168 0.70395216 0.21191593 0.57465003 0.54183902 0.54995229
 0.44224171 0.22423573 0.543045   0.46284407]
theta (2000, 2) [[3.56485228 0.39597168]
 [4.70733526 0.70395216]
 [1.92168994 0.21191593]
 [3.48350921 0.57465003]
 [3.87936542 0.54183902]
 [3.83044543 0.54995229]
 [3.1491516  0.44224171]
 [1.89563379 0.22423573]
 [3.79624839 0.543045  ]
 [2.85325583 0.46284407]]


SystemError: CPUDispatcher(<function _simulator_v4_core at 0x124157c10>) returned a result with an error set

In [12]:
import numpy as np
from numpy.random import default_rng, SeedSequence
from numpy.random import Generator as _NpGen, RandomState as _RS
import functions_list_260305 as functions_list
import summary_stats_elms_260305 as ss
import hashlib
import time
import matplotlib.pyplot as plt


start = time.perf_counter()

# ============================================================================
# SETUP: RNG and parameters
# ============================================================================
rng = np.random.default_rng(123)
core_params_num = 2  # core params: R0 and sigma

# fixed parameters
DurationSimulation = 20.0     # years: 20.0
Nstrains = 40       # number of strains: 42
omega = 0.2     # immunity cross strains: 0.1
x = 10.0        #
Cperweek = 34.53    #
Nagents = 1000      # number of agents
alpha = 0.007        # migration rate: 3.0
AgeDeath = 71.0     # life expectancy

# ============================================================================
# BUILD FIXED PARAMS ARRAY
# ============================================================================
if core_params_num == 2:
    Dimmunity = 10.0 * 52.14  # weeks: 0.5 * 52.14
    fixed_params = np.array([DurationSimulation, Nstrains, Dimmunity, omega,
                         x, Cperweek, Nagents, alpha,
                         AgeDeath], dtype=float)
elif core_params_num == 3:
    fixed_params = np.array([DurationSimulation, Nstrains, omega, x,
                             Cperweek, Nagents, alpha, AgeDeath], dtype=float)
else:
    raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: build_params
# ============================================================================
def build_params(theta, fixed_params, core_params_num):
    theta = np.asarray(theta, float).ravel()
    if theta.size != core_params_num:
        raise ValueError(f"theta must be length-{core_params_num}, got {np.shape(theta)}")
    if core_params_num == 2:
        R0, sigma = float(theta[0]), float(theta[1])
        return np.array([fixed_params[0], fixed_params[1], fixed_params[2], sigma,
                         fixed_params[3], fixed_params[4], fixed_params[5], fixed_params[6],
                         fixed_params[7], fixed_params[8], R0
                         ], dtype=float)
    elif core_params_num == 3:
        R0, sigma, Dimmunity = float(theta[0]), float(theta[1]), float(theta[2])
        return np.array([fixed_params[0], fixed_params[1], Dimmunity, sigma,
                         fixed_params[2], fixed_params[3], fixed_params[4], fixed_params[5],
                         fixed_params[6], fixed_params[7], R0
                         ], dtype=float)
    else:
        raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: seed_from_theta
# ============================================================================
def seed_from_theta(theta, master_seed: int = 123):
    th = np.asarray(theta, np.float64).ravel()
    b  = th.tobytes() + np.uint64(master_seed).tobytes()
    return int.from_bytes(hashlib.sha1(b).digest()[:8], 'little')

# ============================================================================
# FUNCTION: simulate_prevalence_v5_numba
# ============================================================================
def simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed):
    seed = seed_from_theta(theta, master_seed=seed)
    rng = default_rng(seed)
    params = build_params(theta, fixed_params, core_params_num)
    AC, IMM, _ = functions_list.initialise_agents_v5(params, rng=rng)
    SSPrev_selected, SSPrev, AIBKS = functions_list.simulator_v5_numba(
        AC, IMM, params, 0, 1, seed=seed
    )
    return SSPrev_selected.astype(float)

# ============================================================================
# FUNCTION: summary_stats
# ============================================================================
def summary_stats(series_2d):
    y = np.asarray(series_2d, float).ravel()
    avg_prev_obs = ss.avg_prev_numpy(series_2d)
    var_prev_obs = np.sqrt(ss.var_prev_numpy(series_2d))
    avg_npmi_obs = ss.avg_npmi_numpy(series_2d)
    div_all_isolates_obs = ss.div_all_isolates_numpy(series_2d)
    return np.array(
        [avg_prev_obs, var_prev_obs, avg_npmi_obs, div_all_isolates_obs], float)

# ============================================================================
# GENERATE SYNTHETIC DATA
# ============================================================================
if core_params_num == 2:
    _Tdry = simulate_prevalence_v5_numba(np.array([3.5, 0.4], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([3.5, 0.4], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
elif core_params_num == 3:
    _Tdry = simulate_prevalence_v5_numba(np.array([3.5, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([3.5, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
else:
    raise ValueError('Invalid core params num')

y_obs_array = _Tdry
print("y_obs_array.mean:", y_obs_array.mean())
# ============================================================================
# LOAD POSTERIOR SAMPLES
# ============================================================================
csv_path_R0 = "../../experimental_data/from_260312/R0_samps_2params_R03p5.csv"
csv_path_sigma = "../../experimental_data/from_260312/sigma_samps_2params_R03p5.csv"
total_length = 2000
R0_samps = np.loadtxt(csv_path_R0, delimiter=",")
sigma_samps = np.loadtxt(csv_path_sigma, delimiter=",")
R0_samps = np.asarray(R0_samps, dtype=float).ravel()
sigma_samps = np.asarray(sigma_samps, dtype=float).ravel()
theta_samps = np.column_stack((R0_samps, sigma_samps))
theta_samps = theta_samps[np.isfinite(theta_samps).all(axis=1)]
theta_samps = theta_samps[:total_length]
R0_samps = theta_samps[:, 0]
sigma_samps = theta_samps[:, 1]
print("R0", R0_samps.shape, R0_samps[:10])
print("sigma", sigma_samps.shape, sigma_samps[:10])
print("theta", theta_samps.shape, theta_samps[:10, :])

# ============================================================================
# FUNCTION: simulate_at_obs
# ============================================================================
def simulate_at_obs(theta, seed):
    rng = np.random.default_rng(seed)
    Y = simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed=seed)
    return Y

# ============================================================================
# GENERATE POSTERIOR PREDICTIVE CHECKS
# ============================================================================
ppc = []
for m, th in enumerate(theta_samps):
    Y_m = simulate_at_obs(th, seed=123)
    ppc.append(Y_m)
ppc = np.stack(ppc, axis=0)

# ============================================================================
# COMPUTE PREDICTIVE SUMMARIES
# ============================================================================
pred_mean = ppc.mean(axis=0) 
pred_lo, pred_hi = np.quantile(ppc, [0.05, 0.95], axis=0)

# ============================================================================
# FUNCTION: rmse
# ============================================================================
def rmse(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    d = A[mask] - B[mask]
    return float(np.sqrt(np.mean(d*d)))

# ============================================================================
# FUNCTION: mae
# ============================================================================
def mae(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    return float(np.mean(np.abs(A[mask] - B[mask])))

# ============================================================================
# COMPUTE ERROR METRICS
# ============================================================================
rmse_mean = rmse(pred_mean, y_obs_array)
mae_mean  = mae(pred_mean, y_obs_array)
print("rmse:", rmse_mean)
print("mae:", mae_mean)

# ============================================================================
# COMPUTE COVERAGE
# ============================================================================
inside = (y_obs_array >= pred_lo) & (y_obs_array <= pred_hi)
coverage = float(np.mean(inside[np.isfinite(y_obs_array)]))
print("coverage:", coverage)

end = time.perf_counter()
print(f"Elapsed: {end - start:.4f} s")

# ============================================================================
# FUNCTION: heatmap
# ============================================================================
def heatmap(M, title, save_path=None):
    plt.figure(figsize=(7, 4.5))
    plt.imshow(M, aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.xlabel("time index")
    plt.ylabel("strain")
    plt.tight_layout()
    plt.savefig(f"{save_path}{title}.png", dpi=300, bbox_inches="tight")
    plt.show()

# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
heatmap(y_obs_array, "Observed", save_path="../../figures/from_260312/ppc/sigma0p4/R03p5/")
heatmap(pred_mean, "PPC mean", save_path="../../figures/from_260312/ppc/sigma0p4/R03p5/")
heatmap(pred_mean - y_obs_array, "Error (mean - obs)", save_path="../../figures/from_260312/ppc/sigma0p4/R03p5/")

# ============================================================================
# ADDITIONAL DIAGNOSTICS (OPTIONAL)
# ============================================================================
# Uncertainty heatmap
heatmap(pred_hi - pred_lo, "90% CI Width (Uncertainty)", save_path="../../figures/from_260312/ppc/sigma0p4/R03p5/")

# Coverage map
heatmap(inside.astype(float), "Coverage Map (1=inside, 0=outside)", save_path="../../figures/from_260312/ppc/sigma0p4/R03p5/")

# Strain-specific metrics
strain_coverage = inside.mean(axis=1)
strain_mae = np.abs(pred_mean - y_obs_array).mean(axis=1)
print("Worst strain coverage:", strain_coverage.min())
print("Best strain coverage:", strain_coverage.max())
print("Worst strain MAE:", strain_mae.max())

# Time-specific metrics
time_coverage = inside.mean(axis=0)
time_mae = np.abs(pred_mean - y_obs_array).mean(axis=0)
print("Worst time coverage:", time_coverage.min())
print("Best time coverage:", time_coverage.max())
print("Worst time MAE:", time_mae.max())

# ============================================================================
# SUMMARY REPORT
# ============================================================================
print("=" * 60)
print("POSTERIOR PREDICTIVE CHECK RESULTS")
print("=" * 60)
print(f"Ground truth:      R0 = 3.5, sigma = 0.4")
print(f"Posterior samples: {len(theta_samps)}")
print(f"R0 posterior:      {R0_samps.mean():.3f} ± {R0_samps.std():.3f}")
print(f"sigma posterior:   {sigma_samps.mean():.3f} ± {sigma_samps.std():.3f}")
print("-" * 60)
print(f"RMSE:              {rmse_mean:.6f}")
print(f"MAE:               {mae_mean:.6f}")
print(f"90% CI Coverage:   {coverage*100:.2f}%")
print(f"Expected Coverage: 90.00%")
print("-" * 60)
if 0.88 <= coverage <= 0.92:
    print("Status: ✓ Well-calibrated uncertainty")
elif coverage < 0.88:
    print("Status: ⚠ Under-coverage (model overconfident)")
else:
    print("Status: ⚠ Over-coverage (model underconfident)")
print("=" * 60)

T's size 920
True True
y_obs_array.mean: 3.377173913043478
R0 (2000,) [4.07782273 5.44880231 5.79475077 6.51160295 6.01061408 4.39653452
 3.37692667 3.57898192 4.35549806 3.22390699]
sigma (2000,) [0.39597168 0.70395216 0.61453203 0.7004272  0.693193   0.54995229
 0.21778823 0.44224171 0.543045   0.46284407]
theta (2000, 2) [[4.07782273 0.39597168]
 [5.44880231 0.70395216]
 [5.79475077 0.61453203]
 [6.51160295 0.7004272 ]
 [6.01061408 0.693193  ]
 [4.39653452 0.54995229]
 [3.37692667 0.21778823]
 [3.57898192 0.44224171]
 [4.35549806 0.543045  ]
 [3.22390699 0.46284407]]


SystemError: CPUDispatcher(<function _simulator_v4_core at 0x124157c10>) returned a result with an error set

In [13]:
import numpy as np
from numpy.random import default_rng, SeedSequence
from numpy.random import Generator as _NpGen, RandomState as _RS
import functions_list_260305 as functions_list
import summary_stats_elms_260305 as ss
import hashlib
import time
import matplotlib.pyplot as plt


start = time.perf_counter()

# ============================================================================
# SETUP: RNG and parameters
# ============================================================================
rng = np.random.default_rng(123)
core_params_num = 2  # core params: R0 and sigma

# fixed parameters
DurationSimulation = 20.0     # years: 20.0
Nstrains = 40       # number of strains: 42
omega = 0.2     # immunity cross strains: 0.1
x = 10.0        #
Cperweek = 34.53    #
Nagents = 1000      # number of agents
alpha = 0.007        # migration rate: 3.0
AgeDeath = 71.0     # life expectancy

# ============================================================================
# BUILD FIXED PARAMS ARRAY
# ============================================================================
if core_params_num == 2:
    Dimmunity = 10.0 * 52.14  # weeks: 0.5 * 52.14
    fixed_params = np.array([DurationSimulation, Nstrains, Dimmunity, omega,
                         x, Cperweek, Nagents, alpha,
                         AgeDeath], dtype=float)
elif core_params_num == 3:
    fixed_params = np.array([DurationSimulation, Nstrains, omega, x,
                             Cperweek, Nagents, alpha, AgeDeath], dtype=float)
else:
    raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: build_params
# ============================================================================
def build_params(theta, fixed_params, core_params_num):
    theta = np.asarray(theta, float).ravel()
    if theta.size != core_params_num:
        raise ValueError(f"theta must be length-{core_params_num}, got {np.shape(theta)}")
    if core_params_num == 2:
        R0, sigma = float(theta[0]), float(theta[1])
        return np.array([fixed_params[0], fixed_params[1], fixed_params[2], sigma,
                         fixed_params[3], fixed_params[4], fixed_params[5], fixed_params[6],
                         fixed_params[7], fixed_params[8], R0
                         ], dtype=float)
    elif core_params_num == 3:
        R0, sigma, Dimmunity = float(theta[0]), float(theta[1]), float(theta[2])
        return np.array([fixed_params[0], fixed_params[1], Dimmunity, sigma,
                         fixed_params[2], fixed_params[3], fixed_params[4], fixed_params[5],
                         fixed_params[6], fixed_params[7], R0
                         ], dtype=float)
    else:
        raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: seed_from_theta
# ============================================================================
def seed_from_theta(theta, master_seed: int = 123):
    th = np.asarray(theta, np.float64).ravel()
    b  = th.tobytes() + np.uint64(master_seed).tobytes()
    return int.from_bytes(hashlib.sha1(b).digest()[:8], 'little')

# ============================================================================
# FUNCTION: simulate_prevalence_v5_numba
# ============================================================================
def simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed):
    seed = seed_from_theta(theta, master_seed=seed)
    rng = default_rng(seed)
    params = build_params(theta, fixed_params, core_params_num)
    AC, IMM, _ = functions_list.initialise_agents_v5(params, rng=rng)
    SSPrev_selected, SSPrev, AIBKS = functions_list.simulator_v5_numba(
        AC, IMM, params, 0, 1, seed=seed
    )
    return SSPrev_selected.astype(float)

# ============================================================================
# FUNCTION: summary_stats
# ============================================================================
def summary_stats(series_2d):
    y = np.asarray(series_2d, float).ravel()
    avg_prev_obs = ss.avg_prev_numpy(series_2d)
    var_prev_obs = np.sqrt(ss.var_prev_numpy(series_2d))
    avg_npmi_obs = ss.avg_npmi_numpy(series_2d)
    div_all_isolates_obs = ss.div_all_isolates_numpy(series_2d)
    return np.array(
        [avg_prev_obs, var_prev_obs, avg_npmi_obs, div_all_isolates_obs], float)

# ============================================================================
# GENERATE SYNTHETIC DATA
# ============================================================================
if core_params_num == 2:
    _Tdry = simulate_prevalence_v5_numba(np.array([4.0, 0.4], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([4.0, 0.4], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
elif core_params_num == 3:
    _Tdry = simulate_prevalence_v5_numba(np.array([4.0, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([4.0, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
else:
    raise ValueError('Invalid core params num')

y_obs_array = _Tdry
print("y_obs_array.mean:", y_obs_array.mean())
# ============================================================================
# LOAD POSTERIOR SAMPLES
# ============================================================================
csv_path_R0 = "../../experimental_data/from_260312/R0_samps_2params_R04p0.csv"
csv_path_sigma = "../../experimental_data/from_260312/sigma_samps_2params_R04p0.csv"
total_length = 2000
R0_samps = np.loadtxt(csv_path_R0, delimiter=",")
sigma_samps = np.loadtxt(csv_path_sigma, delimiter=",")
R0_samps = np.asarray(R0_samps, dtype=float).ravel()
sigma_samps = np.asarray(sigma_samps, dtype=float).ravel()
theta_samps = np.column_stack((R0_samps, sigma_samps))
theta_samps = theta_samps[np.isfinite(theta_samps).all(axis=1)]
theta_samps = theta_samps[:total_length]
R0_samps = theta_samps[:, 0]
sigma_samps = theta_samps[:, 1]
print("R0", R0_samps.shape, R0_samps[:10])
print("sigma", sigma_samps.shape, sigma_samps[:10])
print("theta", theta_samps.shape, theta_samps[:10, :])

# ============================================================================
# FUNCTION: simulate_at_obs
# ============================================================================
def simulate_at_obs(theta, seed):
    rng = np.random.default_rng(seed)
    Y = simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed=seed)
    return Y

# ============================================================================
# GENERATE POSTERIOR PREDICTIVE CHECKS
# ============================================================================
ppc = []
for m, th in enumerate(theta_samps):
    Y_m = simulate_at_obs(th, seed=123)
    ppc.append(Y_m)
ppc = np.stack(ppc, axis=0)

# ============================================================================
# COMPUTE PREDICTIVE SUMMARIES
# ============================================================================
pred_mean = ppc.mean(axis=0) 
pred_lo, pred_hi = np.quantile(ppc, [0.05, 0.95], axis=0)

# ============================================================================
# FUNCTION: rmse
# ============================================================================
def rmse(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    d = A[mask] - B[mask]
    return float(np.sqrt(np.mean(d*d)))

# ============================================================================
# FUNCTION: mae
# ============================================================================
def mae(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    return float(np.mean(np.abs(A[mask] - B[mask])))

# ============================================================================
# COMPUTE ERROR METRICS
# ============================================================================
rmse_mean = rmse(pred_mean, y_obs_array)
mae_mean  = mae(pred_mean, y_obs_array)
print("rmse:", rmse_mean)
print("mae:", mae_mean)

# ============================================================================
# COMPUTE COVERAGE
# ============================================================================
inside = (y_obs_array >= pred_lo) & (y_obs_array <= pred_hi)
coverage = float(np.mean(inside[np.isfinite(y_obs_array)]))
print("coverage:", coverage)

end = time.perf_counter()
print(f"Elapsed: {end - start:.4f} s")

# ============================================================================
# FUNCTION: heatmap
# ============================================================================
def heatmap(M, title, save_path=None):
    plt.figure(figsize=(7, 4.5))
    plt.imshow(M, aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.xlabel("time index")
    plt.ylabel("strain")
    plt.tight_layout()
    plt.savefig(f"{save_path}{title}.png", dpi=300, bbox_inches="tight")
    plt.show()

# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
heatmap(y_obs_array, "Observed", save_path="../../figures/from_260312/ppc/sigma0p4/R04p0/")
heatmap(pred_mean, "PPC mean", save_path="../../figures/from_260312/ppc/sigma0p4/R04p0/")
heatmap(pred_mean - y_obs_array, "Error (mean - obs)", save_path="../../figures/from_260312/ppc/sigma0p4/R04p0/")

# ============================================================================
# ADDITIONAL DIAGNOSTICS (OPTIONAL)
# ============================================================================
# Uncertainty heatmap
heatmap(pred_hi - pred_lo, "90% CI Width (Uncertainty)", save_path="../../figures/from_260312/ppc/sigma0p4/R04p0/")

# Coverage map
heatmap(inside.astype(float), "Coverage Map (1=inside, 0=outside)", save_path="../../figures/from_260312/ppc/sigma0p4/R04p0/")

# Strain-specific metrics
strain_coverage = inside.mean(axis=1)
strain_mae = np.abs(pred_mean - y_obs_array).mean(axis=1)
print("Worst strain coverage:", strain_coverage.min())
print("Best strain coverage:", strain_coverage.max())
print("Worst strain MAE:", strain_mae.max())

# Time-specific metrics
time_coverage = inside.mean(axis=0)
time_mae = np.abs(pred_mean - y_obs_array).mean(axis=0)
print("Worst time coverage:", time_coverage.min())
print("Best time coverage:", time_coverage.max())
print("Worst time MAE:", time_mae.max())

# ============================================================================
# SUMMARY REPORT
# ============================================================================
print("=" * 60)
print("POSTERIOR PREDICTIVE CHECK RESULTS")
print("=" * 60)
print(f"Ground truth:      R0 = 4.0, sigma = 0.4")
print(f"Posterior samples: {len(theta_samps)}")
print(f"R0 posterior:      {R0_samps.mean():.3f} ± {R0_samps.std():.3f}")
print(f"sigma posterior:   {sigma_samps.mean():.3f} ± {sigma_samps.std():.3f}")
print("-" * 60)
print(f"RMSE:              {rmse_mean:.6f}")
print(f"MAE:               {mae_mean:.6f}")
print(f"90% CI Coverage:   {coverage*100:.2f}%")
print(f"Expected Coverage: 90.00%")
print("-" * 60)
if 0.88 <= coverage <= 0.92:
    print("Status: ✓ Well-calibrated uncertainty")
elif coverage < 0.88:
    print("Status: ⚠ Under-coverage (model overconfident)")
else:
    print("Status: ⚠ Over-coverage (model underconfident)")
print("=" * 60)

T's size 920
True True
y_obs_array.mean: 3.9597826086956522
R0 (2000,) [2.54251911 6.73828193 4.48452278 4.2979326  7.41985801 4.49640725
 3.77308111 4.28545552 4.39948982 3.59455816]
sigma (2000,) [0.31062309 0.73393562 0.54963478 0.636946   0.71881415 0.65528619
 0.21334117 0.57413504 0.51146909 0.39713305]
theta (2000, 2) [[2.54251911 0.31062309]
 [6.73828193 0.73393562]
 [4.48452278 0.54963478]
 [4.2979326  0.636946  ]
 [7.41985801 0.71881415]
 [4.49640725 0.65528619]
 [3.77308111 0.21334117]
 [4.28545552 0.57413504]
 [4.39948982 0.51146909]
 [3.59455816 0.39713305]]


SystemError: CPUDispatcher(<function _simulator_v4_core at 0x124157c10>) returned a result with an error set

In [14]:
import numpy as np
from numpy.random import default_rng, SeedSequence
from numpy.random import Generator as _NpGen, RandomState as _RS
import functions_list_260305 as functions_list
import summary_stats_elms_260305 as ss
import hashlib
import time
import matplotlib.pyplot as plt


start = time.perf_counter()

# ============================================================================
# SETUP: RNG and parameters
# ============================================================================
rng = np.random.default_rng(123)
core_params_num = 2  # core params: R0 and sigma

# fixed parameters
DurationSimulation = 20.0     # years: 20.0
Nstrains = 40       # number of strains: 42
omega = 0.2     # immunity cross strains: 0.1
x = 10.0        #
Cperweek = 34.53    #
Nagents = 1000      # number of agents
alpha = 0.007        # migration rate: 3.0
AgeDeath = 71.0     # life expectancy

# ============================================================================
# BUILD FIXED PARAMS ARRAY
# ============================================================================
if core_params_num == 2:
    Dimmunity = 10.0 * 52.14  # weeks: 0.5 * 52.14
    fixed_params = np.array([DurationSimulation, Nstrains, Dimmunity, omega,
                         x, Cperweek, Nagents, alpha,
                         AgeDeath], dtype=float)
elif core_params_num == 3:
    fixed_params = np.array([DurationSimulation, Nstrains, omega, x,
                             Cperweek, Nagents, alpha, AgeDeath], dtype=float)
else:
    raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: build_params
# ============================================================================
def build_params(theta, fixed_params, core_params_num):
    theta = np.asarray(theta, float).ravel()
    if theta.size != core_params_num:
        raise ValueError(f"theta must be length-{core_params_num}, got {np.shape(theta)}")
    if core_params_num == 2:
        R0, sigma = float(theta[0]), float(theta[1])
        return np.array([fixed_params[0], fixed_params[1], fixed_params[2], sigma,
                         fixed_params[3], fixed_params[4], fixed_params[5], fixed_params[6],
                         fixed_params[7], fixed_params[8], R0
                         ], dtype=float)
    elif core_params_num == 3:
        R0, sigma, Dimmunity = float(theta[0]), float(theta[1]), float(theta[2])
        return np.array([fixed_params[0], fixed_params[1], Dimmunity, sigma,
                         fixed_params[2], fixed_params[3], fixed_params[4], fixed_params[5],
                         fixed_params[6], fixed_params[7], R0
                         ], dtype=float)
    else:
        raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: seed_from_theta
# ============================================================================
def seed_from_theta(theta, master_seed: int = 123):
    th = np.asarray(theta, np.float64).ravel()
    b  = th.tobytes() + np.uint64(master_seed).tobytes()
    return int.from_bytes(hashlib.sha1(b).digest()[:8], 'little')

# ============================================================================
# FUNCTION: simulate_prevalence_v5_numba
# ============================================================================
def simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed):
    seed = seed_from_theta(theta, master_seed=seed)
    rng = default_rng(seed)
    params = build_params(theta, fixed_params, core_params_num)
    AC, IMM, _ = functions_list.initialise_agents_v5(params, rng=rng)
    SSPrev_selected, SSPrev, AIBKS = functions_list.simulator_v5_numba(
        AC, IMM, params, 0, 1, seed=seed
    )
    return SSPrev_selected.astype(float)

# ============================================================================
# FUNCTION: summary_stats
# ============================================================================
def summary_stats(series_2d):
    y = np.asarray(series_2d, float).ravel()
    avg_prev_obs = ss.avg_prev_numpy(series_2d)
    var_prev_obs = np.sqrt(ss.var_prev_numpy(series_2d))
    avg_npmi_obs = ss.avg_npmi_numpy(series_2d)
    div_all_isolates_obs = ss.div_all_isolates_numpy(series_2d)
    return np.array(
        [avg_prev_obs, var_prev_obs, avg_npmi_obs, div_all_isolates_obs], float)

# ============================================================================
# GENERATE SYNTHETIC DATA
# ============================================================================
if core_params_num == 2:
    _Tdry = simulate_prevalence_v5_numba(np.array([4.5, 0.4], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([4.5, 0.4], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
elif core_params_num == 3:
    _Tdry = simulate_prevalence_v5_numba(np.array([4.5, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([4.5, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
else:
    raise ValueError('Invalid core params num')

y_obs_array = _Tdry
print("y_obs_array.mean:", y_obs_array.mean())
# ============================================================================
# LOAD POSTERIOR SAMPLES
# ============================================================================
csv_path_R0 = "../../experimental_data/from_260312/R0_samps_2params_R04p5.csv"
csv_path_sigma = "../../experimental_data/from_260312/sigma_samps_2params_R04p5.csv"
total_length = 2000
R0_samps = np.loadtxt(csv_path_R0, delimiter=",")
sigma_samps = np.loadtxt(csv_path_sigma, delimiter=",")
R0_samps = np.asarray(R0_samps, dtype=float).ravel()
sigma_samps = np.asarray(sigma_samps, dtype=float).ravel()
theta_samps = np.column_stack((R0_samps, sigma_samps))
theta_samps = theta_samps[np.isfinite(theta_samps).all(axis=1)]
theta_samps = theta_samps[:total_length]
R0_samps = theta_samps[:, 0]
sigma_samps = theta_samps[:, 1]
print("R0", R0_samps.shape, R0_samps[:10])
print("sigma", sigma_samps.shape, sigma_samps[:10])
print("theta", theta_samps.shape, theta_samps[:10, :])

# ============================================================================
# FUNCTION: simulate_at_obs
# ============================================================================
def simulate_at_obs(theta, seed):
    rng = np.random.default_rng(seed)
    Y = simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed=seed)
    return Y

# ============================================================================
# GENERATE POSTERIOR PREDICTIVE CHECKS
# ============================================================================
ppc = []
for m, th in enumerate(theta_samps):
    Y_m = simulate_at_obs(th, seed=123)
    ppc.append(Y_m)
ppc = np.stack(ppc, axis=0)

# ============================================================================
# COMPUTE PREDICTIVE SUMMARIES
# ============================================================================
pred_mean = ppc.mean(axis=0) 
pred_lo, pred_hi = np.quantile(ppc, [0.05, 0.95], axis=0)

# ============================================================================
# FUNCTION: rmse
# ============================================================================
def rmse(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    d = A[mask] - B[mask]
    return float(np.sqrt(np.mean(d*d)))

# ============================================================================
# FUNCTION: mae
# ============================================================================
def mae(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    return float(np.mean(np.abs(A[mask] - B[mask])))

# ============================================================================
# COMPUTE ERROR METRICS
# ============================================================================
rmse_mean = rmse(pred_mean, y_obs_array)
mae_mean  = mae(pred_mean, y_obs_array)
print("rmse:", rmse_mean)
print("mae:", mae_mean)

# ============================================================================
# COMPUTE COVERAGE
# ============================================================================
inside = (y_obs_array >= pred_lo) & (y_obs_array <= pred_hi)
coverage = float(np.mean(inside[np.isfinite(y_obs_array)]))
print("coverage:", coverage)

end = time.perf_counter()
print(f"Elapsed: {end - start:.4f} s")

# ============================================================================
# FUNCTION: heatmap
# ============================================================================
def heatmap(M, title, save_path=None):
    plt.figure(figsize=(7, 4.5))
    plt.imshow(M, aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.xlabel("time index")
    plt.ylabel("strain")
    plt.tight_layout()
    plt.savefig(f"{save_path}{title}.png", dpi=300, bbox_inches="tight")
    plt.show()

# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
heatmap(y_obs_array, "Observed", save_path="../../figures/from_260312/ppc/sigma0p4/R04p5/")
heatmap(pred_mean, "PPC mean", save_path="../../figures/from_260312/ppc/sigma0p4/R04p5/")
heatmap(pred_mean - y_obs_array, "Error (mean - obs)", save_path="../../figures/from_260312/ppc/sigma0p4/R04p5/")

# ============================================================================
# ADDITIONAL DIAGNOSTICS (OPTIONAL)
# ============================================================================
# Uncertainty heatmap
heatmap(pred_hi - pred_lo, "90% CI Width (Uncertainty)", save_path="../../figures/from_260312/ppc/sigma0p4/R04p5/")

# Coverage map
heatmap(inside.astype(float), "Coverage Map (1=inside, 0=outside)", save_path="../../figures/from_260312/ppc/sigma0p4/R04p5/")

# Strain-specific metrics
strain_coverage = inside.mean(axis=1)
strain_mae = np.abs(pred_mean - y_obs_array).mean(axis=1)
print("Worst strain coverage:", strain_coverage.min())
print("Best strain coverage:", strain_coverage.max())
print("Worst strain MAE:", strain_mae.max())

# Time-specific metrics
time_coverage = inside.mean(axis=0)
time_mae = np.abs(pred_mean - y_obs_array).mean(axis=0)
print("Worst time coverage:", time_coverage.min())
print("Best time coverage:", time_coverage.max())
print("Worst time MAE:", time_mae.max())

# ============================================================================
# SUMMARY REPORT
# ============================================================================
print("=" * 60)
print("POSTERIOR PREDICTIVE CHECK RESULTS")
print("=" * 60)
print(f"Ground truth:      R0 = 4.5, sigma = 0.4")
print(f"Posterior samples: {len(theta_samps)}")
print(f"R0 posterior:      {R0_samps.mean():.3f} ± {R0_samps.std():.3f}")
print(f"sigma posterior:   {sigma_samps.mean():.3f} ± {sigma_samps.std():.3f}")
print("-" * 60)
print(f"RMSE:              {rmse_mean:.6f}")
print(f"MAE:               {mae_mean:.6f}")
print(f"90% CI Coverage:   {coverage*100:.2f}%")
print(f"Expected Coverage: 90.00%")
print("-" * 60)
if 0.88 <= coverage <= 0.92:
    print("Status: ✓ Well-calibrated uncertainty")
elif coverage < 0.88:
    print("Status: ⚠ Under-coverage (model overconfident)")
else:
    print("Status: ⚠ Over-coverage (model underconfident)")
print("=" * 60)

T's size 920
True True
y_obs_array.mean: 4.511956521739131
R0 (2000,) [7.55803649 4.98231175 8.33698058 7.37634843 4.97361473 4.16923556
 4.75480631 4.88513122 5.88115863 4.43864256]
sigma (2000,) [0.73393562 0.54963478 0.71881415 0.71913303 0.48098752 0.21334117
 0.57413504 0.51146909 0.64632746 0.38168128]
theta (2000, 2) [[7.55803649 0.73393562]
 [4.98231175 0.54963478]
 [8.33698058 0.71881415]
 [7.37634843 0.71913303]
 [4.97361473 0.48098752]
 [4.16923556 0.21334117]
 [4.75480631 0.57413504]
 [4.88513122 0.51146909]
 [5.88115863 0.64632746]
 [4.43864256 0.38168128]]


SystemError: CPUDispatcher(<function _simulator_v4_core at 0x124157c10>) returned a result with an error set

In [15]:
import numpy as np
from numpy.random import default_rng, SeedSequence
from numpy.random import Generator as _NpGen, RandomState as _RS
import functions_list_260305 as functions_list
import summary_stats_elms_260305 as ss
import hashlib
import time
import matplotlib.pyplot as plt


start = time.perf_counter()

# ============================================================================
# SETUP: RNG and parameters
# ============================================================================
rng = np.random.default_rng(123)
core_params_num = 2  # core params: R0 and sigma

# fixed parameters
DurationSimulation = 20.0     # years: 20.0
Nstrains = 40       # number of strains: 42
omega = 0.2     # immunity cross strains: 0.1
x = 10.0        #
Cperweek = 34.53    #
Nagents = 1000      # number of agents
alpha = 0.007        # migration rate: 3.0
AgeDeath = 71.0     # life expectancy

# ============================================================================
# BUILD FIXED PARAMS ARRAY
# ============================================================================
if core_params_num == 2:
    Dimmunity = 10.0 * 52.14  # weeks: 0.5 * 52.14
    fixed_params = np.array([DurationSimulation, Nstrains, Dimmunity, omega,
                         x, Cperweek, Nagents, alpha,
                         AgeDeath], dtype=float)
elif core_params_num == 3:
    fixed_params = np.array([DurationSimulation, Nstrains, omega, x,
                             Cperweek, Nagents, alpha, AgeDeath], dtype=float)
else:
    raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: build_params
# ============================================================================
def build_params(theta, fixed_params, core_params_num):
    theta = np.asarray(theta, float).ravel()
    if theta.size != core_params_num:
        raise ValueError(f"theta must be length-{core_params_num}, got {np.shape(theta)}")
    if core_params_num == 2:
        R0, sigma = float(theta[0]), float(theta[1])
        return np.array([fixed_params[0], fixed_params[1], fixed_params[2], sigma,
                         fixed_params[3], fixed_params[4], fixed_params[5], fixed_params[6],
                         fixed_params[7], fixed_params[8], R0
                         ], dtype=float)
    elif core_params_num == 3:
        R0, sigma, Dimmunity = float(theta[0]), float(theta[1]), float(theta[2])
        return np.array([fixed_params[0], fixed_params[1], Dimmunity, sigma,
                         fixed_params[2], fixed_params[3], fixed_params[4], fixed_params[5],
                         fixed_params[6], fixed_params[7], R0
                         ], dtype=float)
    else:
        raise ValueError('Invalid core params num')

# ============================================================================
# FUNCTION: seed_from_theta
# ============================================================================
def seed_from_theta(theta, master_seed: int = 123):
    th = np.asarray(theta, np.float64).ravel()
    b  = th.tobytes() + np.uint64(master_seed).tobytes()
    return int.from_bytes(hashlib.sha1(b).digest()[:8], 'little')

# ============================================================================
# FUNCTION: simulate_prevalence_v5_numba
# ============================================================================
def simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed):
    seed = seed_from_theta(theta, master_seed=seed)
    rng = default_rng(seed)
    params = build_params(theta, fixed_params, core_params_num)
    AC, IMM, _ = functions_list.initialise_agents_v5(params, rng=rng)
    SSPrev_selected, SSPrev, AIBKS = functions_list.simulator_v5_numba(
        AC, IMM, params, 0, 1, seed=seed
    )
    return SSPrev_selected.astype(float)

# ============================================================================
# FUNCTION: summary_stats
# ============================================================================
def summary_stats(series_2d):
    y = np.asarray(series_2d, float).ravel()
    avg_prev_obs = ss.avg_prev_numpy(series_2d)
    var_prev_obs = np.sqrt(ss.var_prev_numpy(series_2d))
    avg_npmi_obs = ss.avg_npmi_numpy(series_2d)
    div_all_isolates_obs = ss.div_all_isolates_numpy(series_2d)
    return np.array(
        [avg_prev_obs, var_prev_obs, avg_npmi_obs, div_all_isolates_obs], float)

# ============================================================================
# GENERATE SYNTHETIC DATA
# ============================================================================
if core_params_num == 2:
    _Tdry = simulate_prevalence_v5_numba(np.array([5.0, 0.4], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([5.0, 0.4], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
elif core_params_num == 3:
    _Tdry = simulate_prevalence_v5_numba(np.array([5.0, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    T = _Tdry.size
    print("T's size", T)
    _Tdry1 = simulate_prevalence_v5_numba(np.array([5.0, 0.4, 0.25 * 52.14], float), fixed_params, core_params_num, seed=int(123))
    print(np.allclose(_Tdry, _Tdry1), _Tdry.shape == _Tdry1.shape)
else:
    raise ValueError('Invalid core params num')

y_obs_array = _Tdry
print("y_obs_array.mean:", y_obs_array.mean())
# ============================================================================
# LOAD POSTERIOR SAMPLES
# ============================================================================
csv_path_R0 = "../../experimental_data/from_260312/R0_samps_2params_R05p0.csv"
csv_path_sigma = "../../experimental_data/from_260312/sigma_samps_2params_R05p0.csv"
total_length = 2000
R0_samps = np.loadtxt(csv_path_R0, delimiter=",")
sigma_samps = np.loadtxt(csv_path_sigma, delimiter=",")
R0_samps = np.asarray(R0_samps, dtype=float).ravel()
sigma_samps = np.asarray(sigma_samps, dtype=float).ravel()
theta_samps = np.column_stack((R0_samps, sigma_samps))
theta_samps = theta_samps[np.isfinite(theta_samps).all(axis=1)]
theta_samps = theta_samps[:total_length]
R0_samps = theta_samps[:, 0]
sigma_samps = theta_samps[:, 1]
print("R0", R0_samps.shape, R0_samps[:10])
print("sigma", sigma_samps.shape, sigma_samps[:10])
print("theta", theta_samps.shape, theta_samps[:10, :])

# ============================================================================
# FUNCTION: simulate_at_obs
# ============================================================================
def simulate_at_obs(theta, seed):
    rng = np.random.default_rng(seed)
    Y = simulate_prevalence_v5_numba(theta, fixed_params, core_params_num, seed=seed)
    return Y

# ============================================================================
# GENERATE POSTERIOR PREDICTIVE CHECKS
# ============================================================================
ppc = []
for m, th in enumerate(theta_samps):
    Y_m = simulate_at_obs(th, seed=123)
    ppc.append(Y_m)
ppc = np.stack(ppc, axis=0)

# ============================================================================
# COMPUTE PREDICTIVE SUMMARIES
# ============================================================================
pred_mean = ppc.mean(axis=0) 
pred_lo, pred_hi = np.quantile(ppc, [0.05, 0.95], axis=0)

# ============================================================================
# FUNCTION: rmse
# ============================================================================
def rmse(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    d = A[mask] - B[mask]
    return float(np.sqrt(np.mean(d*d)))

# ============================================================================
# FUNCTION: mae
# ============================================================================
def mae(A, B, mask=None):
    A = np.asarray(A, float)
    B = np.asarray(B, float)
    if mask is None:
        mask = np.isfinite(A) & np.isfinite(B)
    return float(np.mean(np.abs(A[mask] - B[mask])))

# ============================================================================
# COMPUTE ERROR METRICS
# ============================================================================
rmse_mean = rmse(pred_mean, y_obs_array)
mae_mean  = mae(pred_mean, y_obs_array)
print("rmse:", rmse_mean)
print("mae:", mae_mean)

# ============================================================================
# COMPUTE COVERAGE
# ============================================================================
inside = (y_obs_array >= pred_lo) & (y_obs_array <= pred_hi)
coverage = float(np.mean(inside[np.isfinite(y_obs_array)]))
print("coverage:", coverage)

end = time.perf_counter()
print(f"Elapsed: {end - start:.4f} s")

# ============================================================================
# FUNCTION: heatmap
# ============================================================================
def heatmap(M, title, save_path=None):
    plt.figure(figsize=(7, 4.5))
    plt.imshow(M, aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.xlabel("time index")
    plt.ylabel("strain")
    plt.tight_layout()
    plt.savefig(f"{save_path}{title}.png", dpi=300, bbox_inches="tight")
    plt.show()

# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
heatmap(y_obs_array, "Observed", save_path="../../figures/from_260312/ppc/sigma0p4/R05p0/")
heatmap(pred_mean, "PPC mean", save_path="../../figures/from_260312/ppc/sigma0p4/R05p0/")
heatmap(pred_mean - y_obs_array, "Error (mean - obs)", save_path="../../figures/from_260312/ppc/sigma0p4/R05p0/")

# ============================================================================
# ADDITIONAL DIAGNOSTICS (OPTIONAL)
# ============================================================================
# Uncertainty heatmap
heatmap(pred_hi - pred_lo, "90% CI Width (Uncertainty)", save_path="../../figures/from_260312/ppc/sigma0p4/R05p0/")

# Coverage map
heatmap(inside.astype(float), "Coverage Map (1=inside, 0=outside)", save_path="../../figures/from_260312/ppc/sigma0p4/R05p0/")

# Strain-specific metrics
strain_coverage = inside.mean(axis=1)
strain_mae = np.abs(pred_mean - y_obs_array).mean(axis=1)
print("Worst strain coverage:", strain_coverage.min())
print("Best strain coverage:", strain_coverage.max())
print("Worst strain MAE:", strain_mae.max())

# Time-specific metrics
time_coverage = inside.mean(axis=0)
time_mae = np.abs(pred_mean - y_obs_array).mean(axis=0)
print("Worst time coverage:", time_coverage.min())
print("Best time coverage:", time_coverage.max())
print("Worst time MAE:", time_mae.max())

# ============================================================================
# SUMMARY REPORT
# ============================================================================
print("=" * 60)
print("POSTERIOR PREDICTIVE CHECK RESULTS")
print("=" * 60)
print(f"Ground truth:      R0 = 5.0, sigma = 0.4")
print(f"Posterior samples: {len(theta_samps)}")
print(f"R0 posterior:      {R0_samps.mean():.3f} ± {R0_samps.std():.3f}")
print(f"sigma posterior:   {sigma_samps.mean():.3f} ± {sigma_samps.std():.3f}")
print("-" * 60)
print(f"RMSE:              {rmse_mean:.6f}")
print(f"MAE:               {mae_mean:.6f}")
print(f"90% CI Coverage:   {coverage*100:.2f}%")
print(f"Expected Coverage: 90.00%")
print("-" * 60)
if 0.88 <= coverage <= 0.92:
    print("Status: ✓ Well-calibrated uncertainty")
elif coverage < 0.88:
    print("Status: ⚠ Under-coverage (model overconfident)")
else:
    print("Status: ⚠ Over-coverage (model underconfident)")
print("=" * 60)

T's size 920
True True
y_obs_array.mean: 5.05
R0 (2000,) [7.67320347 5.47031658 6.18285776 6.09480178 5.37077263 8.30900899
 6.03324709 6.51377505 7.11481734 5.19524883]
sigma (2000,) [0.57796412 0.48098752 0.45637926 0.46246422 0.51146909 0.67930616
 0.45728375 0.52682924 0.61295555 0.32426574]
theta (2000, 2) [[7.67320347 0.57796412]
 [5.47031658 0.48098752]
 [6.18285776 0.45637926]
 [6.09480178 0.46246422]
 [5.37077263 0.51146909]
 [8.30900899 0.67930616]
 [6.03324709 0.45728375]
 [6.51377505 0.52682924]
 [7.11481734 0.61295555]
 [5.19524883 0.32426574]]


SystemError: CPUDispatcher(<function _simulator_v4_core at 0x124157c10>) returned a result with an error set